# Multi-Plane Segmentation Visualization with Bounding Boxes

Visualize segmentation masks and their extracted 3D bounding boxes from three orthogonal planes.

In [ ]:
# Import segmentation processing utilities
import nibabel as nib
from datasets.preprocessing import mask_to_boxes_3d
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load a sample segmentation file
SEGMENTATION_PATH = './datasets/segmentations/21057.nii'
IMAGES_DIR = './datasets/train_images/10004/21057'

print(f"Loading segmentation from {SEGMENTATION_PATH}...")
seg_nib = nib.load(SEGMENTATION_PATH)
seg_data = seg_nib.get_fdata()

print(f"Original segmentation shape (H, W, D): {seg_data.shape}")

# Apply same transformations as segmentation_analysis.py:
# 1. Rotate segmentation to match image orientation (same as resize_segmentation)
seg_data = np.rot90(seg_data, k=1)

# 2. Flip depth dimension (same as frame_index = num_frames - 1 - position_in_range)
seg_data = seg_data[:, :, ::-1]

# 3. Transpose from (H, W, D) to (D, H, W) to match volume_images format
seg_data = np.transpose(seg_data, (2, 0, 1))

print(f"Transformed segmentation shape (D, H, W): {seg_data.shape}")
print(f"Unique labels: {np.unique(seg_data)}")

# Extract 3D bounding boxes using data pipeline function
# Note: mask_to_boxes_3d expects (D, H, W) or (H, W, D) format - check its implementation
boxes_data = mask_to_boxes_3d(seg_data, min_volume=50)

print(f"\nExtracted {len(boxes_data)} bounding boxes:")
for i, box_info in enumerate(boxes_data):
    print(f"  Box {i+1}: Label {box_info['label']}, Box (normalized): {box_info['box']}")

In [ ]:
# Load JPEG images and stack into volume
from PIL import Image
import os

# jpeg_files = sorted([f for f in os.listdir(IMAGES_DIR) if f.endswith('.jpeg')])
jpeg_files = sorted([f for f in os.listdir(IMAGES_DIR) if f.endswith('.jpeg')], 
                    key=lambda x: int(x.replace('.jpeg', '')))
image_indices = sorted([int(f.replace('.jpeg', '')) for f in jpeg_files])

print(f"Found {len(jpeg_files)} images")
print(f"Image index range: {min(image_indices)} to {max(image_indices)}")

# Load all images
images = []
for jpeg_file in jpeg_files:
    img_path = os.path.join(IMAGES_DIR, jpeg_file)
    img = np.array(Image.open(img_path))
    if len(img.shape) == 2:
        img = np.stack([img] * 3, axis=-1)
    images.append(img)

# Stack into volume and reverse to match segmentation
# volume_images = np.stack(images, axis=0)[::-1]
volume_images = np.stack(images, axis=0)
print(f"Volume shape: {volume_images.shape}")

In [3]:
def visualize_slice_with_segmentation_and_bbox(volume, seg_mask, boxes_data, 
                                                 slice_idx, axis='axial'):
    """
    Visualize a 2D slice with segmentation overlay and bounding boxes.
    
    Args:
        volume: 3D image volume (D, H, W, C)
        seg_mask: 3D segmentation mask (D, H, W)
        boxes_data: List of box dicts from mask_to_boxes_3d
        slice_idx: Index of slice to visualize
        axis: 'axial', 'sagittal', or 'coronal'
    
    Returns:
        2D image with overlays
    """
    D, H, W = seg_mask.shape
    
    # Extract 2D slice based on axis
    if axis == 'axial' or axis == 'transverse':
        # Axial/transverse: view from top (slice along D)
        image_slice = volume[slice_idx, :, :, :]
        seg_slice = seg_mask[slice_idx, :, :]
        # For bbox projection: check if cz overlaps with slice
        def project_box(box):
            cx, cy, cz, w, h, d = box
            # Denormalize
            cz_px = cz * D
            d_px = d * D
            cy_px = cy * H
            h_px = h * H
            cx_px = cx * W
            w_px = w * W
            # Check if box intersects this slice
            if abs(cz_px - slice_idx) < d_px / 2:
                # Return 2D bbox: x_min, y_min, x_max, y_max
                return [
                    cx_px - w_px/2, cy_px - h_px/2,
                    cx_px + w_px/2, cy_px + h_px/2
                ]
            return None
    elif axis == 'sagittal':
        # Sagittal: view from side (slice along W)
        image_slice = volume[:, :, slice_idx, :]
        seg_slice = seg_mask[:, :, slice_idx]
        def project_box(box):
            cx, cy, cz, w, h, d = box
            cx_px = cx * W
            w_px = w * W
            cy_px = cy * H
            h_px = h * H
            cz_px = cz * D
            d_px = d * D
            if abs(cx_px - slice_idx) < w_px / 2:
                return [
                    cy_px - h_px/2, cz_px - d_px/2,
                    cy_px + h_px/2, cz_px + d_px/2
                ]
            return None
    elif axis == 'coronal':
        # Coronal: view from front (slice along H)
        image_slice = volume[:, slice_idx, :, :]
        seg_slice = seg_mask[:, slice_idx, :]
        def project_box(box):
            cx, cy, cz, w, h, d = box
            cy_px = cy * H
            h_px = h * H
            cx_px = cx * W
            w_px = w * W
            cz_px = cz * D
            d_px = d * D
            if abs(cy_px - slice_idx) < h_px / 2:
                return [
                    cx_px - w_px/2, cz_px - d_px/2,
                    cx_px + w_px/2, cz_px + d_px/2
                ]
            return None
    
    # Create overlay
    overlay = image_slice.copy().astype(float)
    
    # Define colors for different labels
    label_colors = {
        1: [255, 0, 0],    # Red
        2: [0, 255, 0],    # Green
        3: [0, 0, 255],    # Blue
        4: [255, 255, 0],  # Yellow
        5: [255, 0, 255],  # Magenta
    }
    
    # Resize segmentation to match image if needed
    if seg_slice.shape != image_slice.shape[:2]:
        seg_slice = cv2.resize(seg_slice, (image_slice.shape[1], image_slice.shape[0]), 
                               interpolation=cv2.INTER_NEAREST)
    
    # Overlay segmentation masks with transparency
    for label in range(1, 6):
        mask_bool = seg_slice == label
        if np.any(mask_bool):
            color = np.array(label_colors.get(label, [128, 128, 128]))
            overlay[mask_bool] = 0.6 * overlay[mask_bool] + 0.4 * color
    
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    
    # Draw bounding boxes
    for box_info in boxes_data:
        box_2d = project_box(box_info['box'])
        if box_2d is not None:
            x1, y1, x2, y2 = [int(coord) for coord in box_2d]
            label = box_info['label']
            color = label_colors.get(label, [255, 255, 255])
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
            # Add label text
            cv2.putText(overlay, f'L{label}', (x1, y1-5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    return overlay

In [ ]:
# Transverse (Axial) Plane Visualization
print("Generating Transverse (Axial) Plane Visualization...")

# Use volume_images dimensions (not seg_data) since they may differ
D_vol, H_vol, W_vol = volume_images.shape[:3]
D_seg, H_seg, W_seg = seg_data.shape
num_samples = 12

print(f"Volume shape: {volume_images.shape[:3]}, Segmentation shape: {seg_data.shape}")

# Resize seg_data to match volume_images if needed
if (D_seg, H_seg, W_seg) != (D_vol, H_vol, W_vol):
    print(f"Resizing segmentation from {seg_data.shape} to {(D_vol, H_vol, W_vol)}...")
    from scipy.ndimage import zoom
    zoom_factors = (D_vol / D_seg, H_vol / H_seg, W_vol / W_seg)
    seg_data_resized = zoom(seg_data, zoom_factors, order=0)  # order=0 for nearest neighbor
    print(f"Resized segmentation shape: {seg_data_resized.shape}")
else:
    seg_data_resized = seg_data

# Uniformly sample slice indices based on volume dimensions
slice_indices = np.linspace(0, D_vol-1, num_samples, dtype=int)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for idx, slice_idx in enumerate(slice_indices):
    ax = axes[idx]
    
    # Get visualization using resized segmentation
    vis_image = visualize_slice_with_segmentation_and_bbox(
        volume_images, seg_data_resized, boxes_data, slice_idx, axis='axial'
    )
    
    ax.imshow(vis_image)
    ax.set_title(f'Slice {slice_idx}/{D_vol-1}', fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Transverse (Axial) Plane: Segmentation + Bounding Boxes', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Transverse plane visualization complete!")

In [ ]:
# Sagittal Plane Visualization
print("Generating Sagittal Plane Visualization...")

# Uniformly sample slice indices along W dimension (using volume dimensions)
slice_indices = np.linspace(0, W_vol-1, num_samples, dtype=int)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for idx, slice_idx in enumerate(slice_indices):
    ax = axes[idx]
    
    # Get visualization using resized segmentation
    vis_image = visualize_slice_with_segmentation_and_bbox(
        volume_images, seg_data_resized, boxes_data, slice_idx, axis='sagittal'
    )
    
    ax.imshow(vis_image)
    ax.set_title(f'Slice {slice_idx}/{W_vol-1}', fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sagittal Plane: Segmentation + Bounding Boxes', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Sagittal plane visualization complete!")

In [ ]:
# Coronal Plane Visualization
print("Generating Coronal Plane Visualization...")

# Uniformly sample slice indices along H dimension (using volume dimensions)
slice_indices = np.linspace(0, H_vol-1, num_samples, dtype=int)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for idx, slice_idx in enumerate(slice_indices):
    ax = axes[idx]
    
    # Get visualization using resized segmentation
    vis_image = visualize_slice_with_segmentation_and_bbox(
        volume_images, seg_data_resized, boxes_data, slice_idx, axis='coronal'
    )
    
    ax.imshow(vis_image)
    ax.set_title(f'Slice {slice_idx}/{H_vol-1}', fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Coronal Plane: Segmentation + Bounding Boxes', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Coronal plane visualization complete!")

In [ ]:
# Summary
print("\n" + "="*60)
print("MULTI-PLANE VISUALIZATION SUMMARY")
print("="*60)
print(f"series ID: 21057")
print(f"Volume shape: {seg_data.shape}")
print(f"Number of bounding boxes extracted: {len(boxes_data)}")
print(f"\nVisualization details:")
print(f"  - 3 orthogonal planes (Transverse, Sagittal, Coronal)")
print(f"  - 12 uniformly sampled slices per plane")
print(f"  - Pixel-level segmentation with color-coded labels")
print(f"  - 3D bounding boxes projected onto 2D slices")
print("="*60)